# Paper15 walk-forward realization · February 2020–April 2026

**75 months / 1,569 trading sessions**, starting NAV at 31 January 2020 close. Four saved RF/XGBoost arms, rolling 5/10-year annual training and monthly RET picks. **Total returns are cumulative over this entire period; CAGR is separately annualized.**

This notebook reads only independently verified saved reports and bounded display artifacts. Run All starts no source-data build, model loading, fit, download or backtest. Daily inputs remain separate for execution and stops. [Method and results](../docs/WALKFORWARD_EXTENSION_BACKTEST_20261027.md).

In [ ]:
from pathlib import Path
import json, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/dmv2").is_dir())
sys.path.insert(0, str(ROOT / "src"))
local = ROOT / "configs/walkforward_extension_backtest.local.json"
settings = json.loads(local.read_text()) if local.exists() else {}
reporting_packages = Path(settings.get("reporting_packages", ROOT / ".reporting"))
if reporting_packages.is_dir():
    sys.path.insert(0, str(reporting_packages))
BACKTEST_RUN = Path(settings.get("output_directory", ROOT / "outputs/walkforward_extension_backtest/20261027-v1"))
from dmv2.extension_backtest_reporting import read_extension_backtest
saved = None
try:
    saved = read_extension_backtest(BACKTEST_RUN)
except FileNotFoundError:
    print("Private verified results are unavailable. No compute started.")
else:
    print(saved["status"], "| complete:", saved["complete_paths"], "/ 48 | withheld:", saved["withheld_paths"])
    print(saved["start_date"], "to", saved["end_date"], "|", saved["months"], "months /", saved["sessions"], "sessions")
    print("Old 47-month controls preserved:", saved["old_prefix_paths_unchanged"], "/ 28")

## Cumulative total return: fixed baskets across execution assumptions

Same-close monthly/daily control; daily next-open gross; daily next-open net at **10 bp one-way**; the same net path with **10% per-asset stops**. These are the Paper15 side experiment, not the earlier expanded-RF arms. No annual NAV resets or terminal liquidation. All 48 cost/stop/control paths remain in the complete table below.

In [ ]:
if saved:
    import pandas as pd
    from IPython.display import display, Image
    results = pd.DataFrame(saved["comparison"])
    stages = ["cache_close_gross", "execution_open_gross", "execution_open_net", "execution_open_net_stop"]
    table = results.loc[results.stage.isin(stages)].pivot(index="arm", columns="stage", values="total_return")[stages]
    display(table.style.format("{:+.2%}", na_rep="WITHHELD").set_caption("Cumulative compounded TOTAL RETURN · February 2020–April 2026 · 75 months"))
    display(Image(filename=saved["figures"]["equity_comparison"]))

## Annualized and daily risk measures

CAGR uses the frozen 252/1,569 daily convention; Sharpe uses daily returns and zero cash rate. Maximum drawdown is measured from daily closes, including the starting NAV. The chart shows daily drawdown for net returns with asset stops. Stops can worsen returns; gap fills and idealized threshold fills are assumptions, not guaranteed executable prices.

In [ ]:
if saved:
    risk = results.loc[results.stage.isin(["execution_open_net", "execution_open_net_stop"])].set_index(["arm", "stage"])[["complete_path", "total_return", "cagr", "daily_sharpe", "daily_volatility", "daily_maximum_drawdown", "stop_count"]]
    display(risk.style.format({"total_return":"{:+.2%}", "cagr":"{:+.2%}", "daily_sharpe":"{:.2f}", "daily_volatility":"{:.2%}", "daily_maximum_drawdown":"{:.2%}"}, na_rep="WITHHELD"))
    display(Image(filename=saved["figures"]["daily_drawdown"]))

## Complete cost sensitivity and continuous annual slices

Costs are 0, 5, 10 and 25 bp per one-way voluntary trade. Zero borrow and financing charges remain an explicit limitation. **2020 has 11 months; 2026 has January–April only.** Annual rows are slices of the continuous portfolio, not independent refits or resets. U.S. Steel’s June 2025 signed cash entitlement affects realized holdings only; labels, RET and saved predictions remain frozen.

In [ ]:
if saved:
    display(results.set_index(["arm", "stage"])[["complete_path", "cost_bps", "asset_stop_fraction", "total_return", "cagr", "daily_sharpe", "daily_maximum_drawdown", "stop_count", "first_failure_date"]])
    annual = pd.DataFrame(saved["annual"])
    display(annual.loc[annual.stage.isin(["execution_open_net", "execution_open_net_stop"])].set_index(["arm", "stage", "year"]).style.format({"total_return":"{:+.2%}"}))
    print("Same-cache monthly/daily maximum absolute month-end NAV difference:", saved["same_cache_same_close_maximum_nav_difference"])
    for limitation in saved["limits"]:
        print("•", limitation)